# 大模型部署 [以 DeepSeek-R1-Distill-Qwen-1.5B 为例]

## 一、环境配置

创建开发环境

![alt text](image.png)

如果使用VS Code  
可以用下载的私钥在vscode上配置settings： 
比如：   
Host mopaas  
  HostName 10.8.160.3  
  User root  
  Port 37125  
  IdentityFile ~/.ssh/private_key_mopass06.pem  

如果使用MoPaaS平台，直接打开终端进行下列操作即可

### 1. 创建虚拟环境
平台已提前为我们安装好适用于海光DCU的pytorch、transformers等工具，因此无需额外构建虚拟环境。

### 2. 下载模型

魔搭社区(modelscope)是一个国内的模型库，比起huggingface能提供更稳定的下载。
从 modelscope 下载要部署的模型文件，本次部署用到 DeepSeek-R1-Distill-Qwen-1.5B (https://www.modelscope.cn/models/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B)。复制信息，填入下载命令中。

In [0]:
# 先安装 modelscope 的包
%pip install modelscope

# 从 modelscope 官网上找到我们要的模型，复制完整模型库的下载命令（如下，但是先不要运行这个）

!modelscope download --model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B

# 然后，我们在上述命令后面，加上一个指定模型存储的路径的指令：

!modelscope download --model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B --local_dir model/DeepSeek-R1-Distill-Qwen-1.5B

# 上述步骤在终端的运行时间大概是半个小时，因为其中包含了模型的原始参数等文件（超过3个G），所以需要的时间比较久，大家耐心等待

如果后续大家想部署其他的大模型，可以自己在modelscope官网（https://www.modelscope.cn/home） 上下载。  

到此，大模型部署的环境配置部分就结束了。

## 二、transformers 库部署模型

我们需要基于刚刚配置好的环境，创建一个新的kennel给Jupyter notebook用

In [0]:
# 在终端，执行以下命令创建一个新的notebook kennel：

!python -m ipykernel install --name base  --display-name base

# --user 指定安装在当前用户的家目录下
# --name deepseek: 指定内核的名称为 deepseek 。这个名称会出现在 Jupyter 笔记本中选择内核的列表中
# -–display-name deepseek :指定内核的显示名称为 deepseek 。这个显示名称会在 Jupyter 笔记本中选择内核时显示

注意，需要你退出当前的Mlab再重新进来，才会显示新的内核  
我们再重新打开当前这个Jupyter文件，然后切换内核为deepseek

从这里开始，我们将不在用终端演示，而是用上我们前面下载过的“ipykernel”，配置好Jupyter notebook的kernel，直接在这里运行

### 2. 加载模型

In [ ]:
# 导入所需的类，AutoTokenizer 将文本切分编号，AutoModelForCausalLM 加载因果语言模型，根据上一句输入预测下一句输出
from transformers import AutoTokenizer, AutoModelForCausalLM

# 指定模型的路径
model_path = "model/DeepSeek-R1-Distill-Qwen-1.5B"

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_path)

# 加载模型
# torch_dtype="auto" 表示自动选择合适的数据类型
# device_map="auto" 表示自动将模型分配到可用的设备上
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto", # float32 float16 bfloat16
    device_map="auto"
)

# 将模型设为评估模式（因为这里我们只需要部署，不需要训练模型），模型会关闭一些不要的功能，以节省显存和加快运行速度
model.eval()

# 查看模型所在的设备和精度
model.device, model.dtype

### 3. 进行对话

In [ ]:
# 定义用户输入
prompt = "1+2等于几"

# 创建消息列表，包含系统消息和用户消息
messages = [
    {"role": "system", "content": "你是一个智能助手，擅长用中文回答用户的提问。"},
    {"role": "user", "content": prompt}
]

# 分词器调用聊天模板将消息转化为模型可接受的格式
# tokenize=False表示不进行标记化，返回文本字符串
# add_generation_prompt=True表示添加生成提示标记
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(text)

# 将格式化后的文本转换为模型输入的张量，并移至模型所在的设备
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
print(model_inputs)

In [ ]:
# 使用模型生成回复
# max_new_tokens=1024 表示限制生成的最大token数为1024
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=1024,
    temperature = 0.7,  # 0 - 1.5      采样温度，介于 0 和 2 之间。更高的值，如 0.8，会使输出更随机，而更低的值，如 0.2，会使其更加集中和确定。 我们通常建议可以更改这个值或者更改 top_p，但不建议同时对两者进行修改
    top_p = 0.8,  # 0.7 - 0.9     作为调节采样温度的替代方案，模型会考虑前 top_p 概率的 token 的结果。所以 0.1 就意味着只有包括在最高 10% 概率中的 token 会被考虑。 我们通常建议修改这个值或者更改 temperature，但不建议同时对两者进行修改。
    repetition_penalty = 1.1  # 1 - 1.3
)

# 从生成的 ID 中提取新生成的部分
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# 将 ID 解码为文本，skip_special_tokens=True 表示跳过特殊标记
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# 打印模型回复
print(response)

### 4. 扩展

#### 1. 自定义输出风格

在model.generate中可设置超参数：  
其中，temperature 和 top_p 是两种不同的控制生成内容随机性的超参数，将这两个参数设置得越低，会得到越稳定保守的结果，反之则会得到更多样和创造性的输出。

repetition_penalty 用于减少文本中的重复内容。值大于 1 时，会降低已经生成过的词汇再次被选中的概率。

通常，建议将取值控制在以下范围：  
temperature：0 - 1.5  
top_p: 0.7 - 0.9  
repetition_penalty: 1 - 1.3  

#### 2. 一个思考

In [ ]:
# 这个时候，如果我们继续提问，由于还没有为模型设计“记忆功能”，它不知道上一轮我们提问了“1+2等于几”，因此，也无法很好的回答下面这个问题

prompt = "什么情况下结果不是3"

messages = [
    {"role": "system", "content": "你是一个智能助手，擅长用中文回答用户的提问。"},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=1024,
    temperature = 0.7,
    top_p = 0.8,
    repetition_penalty = 1.1
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(response)

#### 3. 多轮对话

In [ ]:
# 要实现多轮对话能力，需要保存之前的对话内容
history = []

query = input("请输入内容：")
# query = ["1+2等于几", "什么情况下结果不是3"]
# i = 0

# 输入信息初始化为一个仅包含系统消息的字典
messages = [{"role": "system", "content": "你是一个智能助手，擅长用中文回答用户的提问。"}]

while query:
    # 将历史对话按顺序加入到 messages 中
    for query_h, response_h in history:
        messages.append({"role": "user", "content": query_h})
        messages.append({"role": "assistant", "content": response_h})

    # 加入当前的用户输入
    messages.append({"role": "user", "content": query})
    
    # 输入处理与模型回复解码
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1024,
        temperature = 0.7,
    	top_p = 0.8,
        repetition_penalty=1.1
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(response)
    print("-"*100)
    
    # 将本轮对话的历史加入 history列表中
    history.append([query, response])
    
    # 用户的下一条疑问
    query = input("请输入内容：")
    
    # i += 1
    # if i > len(query)-1:
        # break

## 三、Gradio 搭建网页 chatbot

尽管我们成功地把大模型部署到了本地，但使用命令行的交互方式让人很不舒服，所以我们可以制作一个web页面来跟大模型对话。  
Gradio 是专为机器学习模型设计的 Web 界面制作工具。

In [ ]:
# 安装 Gradio库
%pip install gradio

In [ ]:
# 导入 Gradio库
import gradio as gr

In [ ]:
# 在我们之前写的py文件中，将生成回复写成一个函数并略微修改历史对话记忆，让其符合gradio的格式

def generate_response(query, history, temperature=0.6, top_p=0.95, max_length=1024, repetition_penalty=1.1): 
    # 相同的多轮对话实现
    messages = [{"role": "system", "content": "你是一个智能助手，擅长用中文回答用户的提问。"}]
    for query_h, response_h in history:
        messages.append({"role": "user", "content": query_h})
        messages.append({"role": "assistant", "content": response_h})
    messages.append({"role": "user", "content": query})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_length,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        do_sample=True
    )
    
    response = tokenizer.decode(generated_ids[0][len(model_inputs.input_ids[0]):], skip_special_tokens=True)
    
    # 如果不希望展示思考内容，添加以下代码段处理模型的生成结果
    think_content = "" 
    if "</think>" in response: 
        parts = response.split("</think>", 1) 
        think_content = parts[0].strip() 
        if think_content.startswith("<think>"): 
            think_content = think_content[7:].strip() 
        response = parts[1].strip()

    # 返回模型的回答与思考内容
    return response, think_content

In [ ]:
# 创建事件处理函数：

def clear_history(): 
    return None
    
def respond(message, chat_history, temp, p, length, penalty): 
    chat_history = chat_history + [[message, None]] 
    user_message = chat_history[-1][0] 
    print("temp:", temp) 
    bot_message, _ = generate_response( 
        user_message,  
        chat_history[:-1],  
        temperature=temp, 
        top_p=p, 
        max_length=length, 
        repetition_penalty=penalty 
    ) 
    chat_history[-1][1] = bot_message 
    return "", chat_history

In [ ]:
# 接下来，开始创建gradio界面
# 创建Blocks对象，这是Gradio的主要界面构建工具：

with gr.Blocks(css=".container { max-width: 800px; margin: auto; }") as demo:
    #创建标题 
    gr.HTML( 
            """ 
            <div style="text-align: center; max-width: 800px; margin: 0 auto;"> 
                <div style="display: inline-flex; align-items: center; gap: 0.8rem; font-size: 1.75rem;"> 
                    <h1 style="font-weight: 900; margin-bottom: 7px; margin-top: 5px;"> 
                        DeepSeek 大模型聊天界面 
                    </h1> 
                </div> 
                <p style="margin-bottom: 10px; font-size: 94%; line-height: 23px;"> 
                    基于DeepSeek-R1-1.5B模型的聊天应用 
                </p> 
            </div> 
            """ 
        ) 
 
    with gr.Row():  # 创建行布局 
        with gr.Column(scale=4):  # 创建列布局，scale=4表示占据4/5的宽度 
            # 创建聊天机器人组件 
            chatbot = gr.Chatbot( 
                [],  # 初始对话历史为空 
                elem_id="chatbot", 
                height=600,  # 设置高度 
            ) 
            
            with gr.Row():  # 创建输入区域的行布局 
                with gr.Column(scale=8):  # 输入框占8/9的宽度 
                    # 创建文本输入框 
                    msg = gr.Textbox( 
                        show_label=False,  # 不显示标签 
                        placeholder="请输入您的问题...",  # 占位文本 
                        container=False  # 不显示容器边框 
                    ) 
                submit_btn = gr.Button("发送", scale=1, variant='primary')  # 创建发送按钮 
             
            with gr.Row(): 
                clear_btn = gr.Button("清空对话")  # 创建清空对话按钮 
         
        with gr.Column(scale=1):  # 参数设置区域，占据1/5的宽度 
            with gr.Accordion("参数设置", open=True):  # 创建可折叠的参数设置区域 
                # 创建各种参数的滑动条 
                temperature = gr.Slider( 
                    minimum=0.1,  
                    maximum=2.0,  
                    value=0.6,  
                    step=0.01,  
                    label="Temperature",  
                    info="控制生成文本的随机性" 
                ) 
                top_p = gr.Slider( 
                    minimum=0.5,  
                    maximum=1.0,  
                    value=0.95,  
                    step=0.05,  
                    label="Top P",  
                    info="控制生成文本的多样性" 
                ) 
                max_length = gr.Slider( 
                    minimum=256,  
                    maximum=10240,  
                    value=1024,  
                    step=128,  
                    label="最大生成长度" 
                ) 
                repetition_penalty = gr.Slider( 
                    minimum=1.0,  
                    maximum=1.5,  
                    value=1.1,  
                    step=0.05,  
                    label="重复惩罚系数",  
                    info="控制生成文本的重复度" 
                )
    
    # 创建事件监听器(绑定界面元素与实现函数)
    # 当用户回车时执行respond函数
    msg.submit( 
        respond, 
        [msg, chatbot, temperature, top_p, max_length, repetition_penalty], 
        [msg, chatbot] 
    ) 

    # 当用户点击submit按钮时执行respond函数
    submit_btn.click( 
        respond, 
        [msg, chatbot, temperature, top_p, max_length, repetition_penalty], 
        [msg, chatbot] 
    ) 

    # 当用户点击clear按钮时执行clear_history函数 
    clear_btn.click(clear_history, None, chatbot, queue=False)

In [ ]:
# 启动Gradio应用

if __name__ == "__main__": 
    demo.queue()  # 启用队列功能，用于处理并发请求 
    # demo.launch(server_name="127.0.0.1", server_port=7860, share=False, inbrowser=False)
    demo.launch(share=False, inbrowser=True) # 启动应用，不分享公开链接，自动打开浏览器